# Parametric vs. symbolic verification

Does a trained model solve a held-out query **symbolically** (reads the operands' meaning from
the in-context examples) or **parametrically** (uses memorized token identities)? Accuracy alone
can't separate them, so we relabel tokens two ways and see what breaks:

| intervention | what it does | symbolic | parametric |
|---|---|---|---|
| **context_shuffle** | relabel the symbols in the examples only, leave the question's symbols | breaks (its evidence is now gibberish) | survives (wasn't using examples) |
| **global_relabel** | relabel symbols everywhere (examples + question) consistently — just a rename | survives (rename changes nothing) | breaks (memorized identities no longer match) |

$$\text{symbolic\_reliance} = \frac{\text{acc\_full} - \text{acc\_context\_shuffle}}{\text{acc\_full}} \in [0, 1]$$

where `acc_full` is held-out accuracy on the intact sequence. **≈1 → symbolic, ≈0 → parametric.**

In [ ]:
import torch
from src.load_utils import load_gpt, load_task
from src.device import Compute
from src.readout import symbolic_reliance, relabel_table, final_token_correct

ctx = Compute.resolve('auto')        # CPU locally; CUDA on a T4 / cluster node
DIRNAME = '../outputs/mixrosette-facts-10-16-8heads'
model = load_gpt(DIRNAME, device=ctx.device,
                 disable_flash_attention=ctx.disable_flash_attention).to(ctx.device).eval()
task, task_name = load_task(DIRNAME)
NUM_SYMBOLS, VOCAB_SIZE = task.num_symbols, task.vocab_size
ctx, task_name, NUM_SYMBOLS, VOCAB_SIZE

In [ ]:
# Readout on the released all-variable model (the symbolic reference point).
for k, v in symbolic_reliance(model, task, ctx.device, k_shots=100, fixed_p=0.0).items():
    print(f'{k:20s} {v:.3f}')

## Result on the released model

`acc_full` ≈ 0.95, `acc_context_shuffle` ≈ 0.11 (collapses), `acc_global_relabel` ≈ 0.96
(invariant) → **`symbolic_reliance` ≈ 0.885**. The released all-variable model is **symbolic**.

## `fixed_p` sweep and per-operand-class slicing

`fixed_p` pins a fraction of element-types to a *canonical, cross-sequence* token
(memorizable) while the rest stay per-sequence random.

1. **Sweep** — does presenting more tokens as fixed lower `symbolic_reliance`? An
   all-variable model never learned a parametric route, so reliance stays high at every
   `fixed_p` (even 1.0).
2. **Slice by operand fixity** — split queries by whether both operands are fixed vs variable
   tokens (via `fixed_vocabulary`). A parametric model would show fixed-operand reliance → 0.

In [ ]:
print(f"{'fixed_p':>8} {'acc_full':>9} {'acc_ctx':>8} {'acc_glob':>9} {'reliance':>9}")
for fp in [0.0, 0.25, 0.5, 0.75, 1.0]:
    s = symbolic_reliance(model, task, ctx.device, k_shots=80, fixed_p=fp)
    print(f"{fp:>8.2f} {s['acc_full']:>9.3f} {s['acc_context_shuffle']:>8.3f} "
          f"{s['acc_global_relabel']:>9.3f} {s['symbolic_reliance']:>9.3f}")

In [ ]:
# Per-operand-class slicing at fixed_p=0.5, reusing the shared readout helpers.
B = 256
batch = task.sample_batch(batch_size=B, k_shots=80, max_length=model.config.block_size,
                          hold_out=True, fixed_p=0.5)
inp, tgt = batch['inputs'], batch['targets']
ctx_len = inp.size(1) - 4
ctx_shuf = inp.clone()
ctx_shuf[:, :ctx_len] = torch.gather(relabel_table(B, NUM_SYMBOLS, VOCAB_SIZE), 1, inp[:, :ctx_len])
full = final_token_correct(model, inp, tgt, ctx.device)
shuf = final_token_correct(model, ctx_shuf, tgt, ctx.device)

a_id, b_id = inp[:, -3], inp[:, -2]                       # query operands in ',a b ='
fixed_ids = [{task.numfor[c] for c in fv} for fv in batch['fixed_vocabulary']]
a_fixed = torch.tensor([a_id[i].item() in fixed_ids[i] for i in range(B)])
b_fixed = torch.tensor([b_id[i].item() in fixed_ids[i] for i in range(B)])

for label, mask in [('both-fixed', a_fixed & b_fixed), ('both-variable', ~a_fixed & ~b_fixed)]:
    n = int(mask.sum())
    af, ac = full[mask].mean().item(), shuf[mask].mean().item()
    print(f'{label:>14}: n={n:>3}  acc_full={af:.3f}  acc_ctx={ac:.3f}  reliance={(af - ac) / af:.3f}')

## Result: symbolic (released) vs parametric (trained `fixed_p=1`)

| probe | released (all-variable) | tiny PoC (`fixed_p=1`) |
|---|---|---|
| `acc_context_shuffle` | 0.11 (collapses) | 0.69 (survives) |
| `acc_global_relabel` | 0.96 (invariant) | 0.10 (collapses) |
| **`symbolic_reliance`** | **0.885 (symbolic)** | **0.250 (parametric)** |

Both diagnostic axes flip. On the all-variable model `symbolic_reliance` stays high at every
`fixed_p`; the discriminative split (fixed-operand reliance → 0) emerges once a model is
*trained* on `fixed_p > 0` data (`train_fixed_p.py`). Tracking `symbolic_reliance` over
training — with weight decay — is the grokking-dynamics question.